# Mock Member Health Analysis - Modeling Baseline

## 1. Imports

In [1]:
import os  # Used to create output folders if needed
import pandas as pd  # Used for loading and working with tabular data
import numpy as np  # Used for numerical operations

from sklearn.model_selection import train_test_split  # Used to split data into training and test sets
from sklearn.preprocessing import OneHotEncoder, StandardScaler  # Used for categorical encoding and numeric scaling
from sklearn.compose import ColumnTransformer  # Used to apply different preprocessing to different column types
from sklearn.pipeline import Pipeline  # Used to combine preprocessing and modeling steps
from sklearn.linear_model import LogisticRegression  # Baseline linear classification model
from sklearn.tree import DecisionTreeClassifier  # Baseline nonlinear classification model

from sklearn.metrics import (
    accuracy_score,  # Overall prediction accuracy
    precision_score,  # Share of predicted positives that are truly positive
    recall_score,  # Share of actual positives correctly identified
    f1_score,  # Balance between precision and recall
    roc_auc_score,  # Ability to rank positive cases above negative cases
    confusion_matrix,  # Raw classification error breakdown
    classification_report  # Full precision, recall, and F1 summary
)

## 2.Load Processed Dataset

In [2]:
df = pd.read_csv("../data/processed/member_analysis_ready.csv")
df.head()

,member_id,age,gender,region,plan_type,sdoh_risk_score,chronic_condition_count,engagement_score,pcp_attributed_24mo,prior_awv_count,...,engagement_group,age_group,high_cost_member,chronic_burden_group,sdoh_risk_group,prior_awv_group,total_acute_visits,acute_utilization_group,pcp_status,has_acute_utilization
0,M00001,69,Male,Suburban,Medicare Advantage,40.7,2,68.9,1,1,...,Q4_High,65-79,1,Moderate,Q1_Low,1,2,Low,Attributed,1
1,M00002,32,Female,Urban,DSNP,80.0,3,30.4,0,0,...,Q1_Low,18-34,0,Moderate,Q4_High,0,2,Low,Not Attributed,1
2,M00003,89,Female,Suburban,Medicare Advantage,49.6,3,86.3,1,3,...,Q4_High,80+,0,Moderate,Q2,3,0,NaN,Attributed,0
3,M00004,78,Male,Suburban,Medicare Advantage,45.7,4,63.1,1,1,...,Q4_High,65-79,1,High,Q1_Low,1,3,Moderate,Attributed,1
4,M00005,38,Male,Suburban,Medicare Advantage,32.4,0,55.6,0,0,...,Q4_High,35-49,0,Low,Q1_Low,0,0,NaN,Not Attributed,0


## 3.Define Target and Features

In [3]:
target = "awv_completed"  # Define the binary target variable

drop_cols = [
    "member_id",  # Identifier, not a real predictive feature
    target,  # Target must be removed from predictors
    "monthly_cost",  # Cost outcome should not be used to predict AWV completion
    "high_cost_member"  # Derived from monthly_cost, so also remove to avoid leakage-like behavior
]

X = df.drop(columns=drop_cols)  # Create predictor dataset after removing target and outcome-style variables
y = df[target]  # Store AWV completion target separately

## 4. Prepare Modeling Data

In [4]:
categorical_cols = X.select_dtypes(
    include=["object", "string", "category", "bool"]
).columns.tolist()  # Identify categorical predictors for one-hot encoding

numeric_cols = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()  # Identify numeric predictors for scaling

In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(drop="first", handle_unknown="ignore"),
            categorical_cols
        ),  # Convert categorical variables into dummy variables

        (
            "num",
            StandardScaler(),
            numeric_cols
        )  # Scale numeric variables for logistic regression stability
    ]
)

## 5. Train-Test Split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify = y)

## 6. Build Baseline Models

In [7]:
# Logistic Regression

log_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=3000))
])

log_model.fit(X_train, y_train) 

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [8]:
# Decision Tree
tree_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(max_depth = 4 , random_state=42))  
])

tree_model.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

## 7. Evaluate Model Performance

* accuracy = overall correctness
* precision = how clean are predicted positives
* recall = how many true positives are caught
* F1 = balance of precision and recall
* ROC AUC = ranking ability using probabilities
* confusion matrix = raw error breakdown

In [9]:
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)  # Generate class predictions: 0 or 1
    y_prob = model.predict_proba(X_test)[:, 1]  # Generate predicted probability for class 1

    print(f"\n===== {model_name} =====")
    print("Accuracy:", accuracy_score(y_test, y_pred))  # Overall correctness
    print("Precision:", precision_score(y_test, y_pred, zero_division=0))  # Cleanliness of predicted positives
    print("Recall:", recall_score(y_test, y_pred, zero_division=0))  # Ability to catch actual positives
    print("F1 Score:", f1_score(y_test, y_pred, zero_division=0))  # Balance of precision and recall
    print("ROC AUC:", roc_auc_score(y_test, y_prob))  # Ranking ability across thresholds

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))  # Shows TN, FP, FN, TP

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))  # Full classification metric report
    

In [10]:
evaluate_model(log_model, X_test, y_test, "Logistic Regression")
evaluate_model(tree_model, X_test, y_test, "Decision Tree")


===== Logistic Regression =====
Accuracy: 0.7
Precision: 0.7024539877300614
Recall: 0.7339743589743589
F1 Score: 0.7178683385579937
ROC AUC: 0.7636774394586895

Confusion Matrix:
[[191  97]
 [ 83 229]]

Classification Report:
              precision    recall  f1-score   support

           0       0.70      0.66      0.68       288
           1       0.70      0.73      0.72       312

    accuracy                           0.70       600
   macro avg       0.70      0.70      0.70       600
weighted avg       0.70      0.70      0.70       600


===== Decision Tree =====
Accuracy: 0.66
Precision: 0.6719745222929936
Recall: 0.6762820512820513
F1 Score: 0.6741214057507987
ROC AUC: 0.7238192218660968

Confusion Matrix:
[[185 103]
 [101 211]]

Classification Report:
              precision    recall  f1-score   support

           0       0.65      0.64      0.64       288
           1       0.67      0.68      0.67       312

    accuracy                           0.66       600
   mac

## 8. Save Key Outputs

In [11]:
results = pd.DataFrame({
    "model": ["Logistic Regression", "Decision Tree"],  # Model names

    "accuracy": [
        accuracy_score(y_test, log_model.predict(X_test)),
        accuracy_score(y_test, tree_model.predict(X_test))
    ],  # Overall correctness

    "precision": [
        precision_score(y_test, log_model.predict(X_test), zero_division=0),
        precision_score(y_test, tree_model.predict(X_test), zero_division=0)
    ],  # Positive prediction quality

    "recall": [
        recall_score(y_test, log_model.predict(X_test), zero_division=0),
        recall_score(y_test, tree_model.predict(X_test), zero_division=0)
    ],  # Positive case capture rate

    "f1": [
        f1_score(y_test, log_model.predict(X_test), zero_division=0),
        f1_score(y_test, tree_model.predict(X_test), zero_division=0)
    ],  # Balance of precision and recall

    "roc_auc": [
        roc_auc_score(y_test, log_model.predict_proba(X_test)[:, 1]),
        roc_auc_score(y_test, tree_model.predict_proba(X_test)[:, 1])
    ]  # Ranking performance
})

In [12]:
results.sort_values(
    by=["roc_auc", "f1"],  # Rank models by ROC AUC first, then F1 score
    ascending=False  # Highest scores are better
)

,model,accuracy,precision,recall,f1,roc_auc
0,Logistic Regression,0.70,0.702454,0.733974,0.717868,0.763677
1,Decision Tree,0.66,0.671975,0.676282,0.674121,0.723819


In [13]:
os.makedirs("../outputs", exist_ok=True)  # Create outputs folder if it does not exist

results.to_csv(
    "../outputs/day5_model_comparison.csv",
    index=False
)  # Save model comparison results

## 9. Day 5 Findings

The purpose of this notebook was to build baseline classification models to predict current-year AWV completion.

The target variable was `awv_completed`.

Two baseline models were compared:

1. Logistic Regression
2. Decision Tree Classifier

The predictor set excluded `member_id`, `monthly_cost`, and `high_cost_member`. `member_id` was excluded because it is only an identifier. `monthly_cost` and `high_cost_member` were excluded because they are cost-related outcome variables and should not be used to predict AWV completion in this baseline classification task.

Categorical variables were one-hot encoded, and numeric variables were standardized. The same preprocessing pipeline was used for both models to keep the comparison consistent.

Model performance was evaluated using accuracy, precision, recall, F1 score, ROC AUC, confusion matrix, and classification report.

Accuracy alone is not enough because AWV completion may be imbalanced. Precision, recall, F1 score, and ROC AUC provide a more complete view of classification performance.

### Better Baseline Model: Logistic Regression

Logistic Regression was selected as the better baseline model because it achieved higher ROC AUC (`0.76`) and F1 score (`0.72`) than the Decision Tree (`0.72` ROC AUC, `0.67` F1).

ROC AUC was used to compare ranking ability across probability thresholds, while F1 score was used to compare the balance between precision and recall.

This suggests that Logistic Regression provides a stronger and more stable baseline for predicting AWV completion in this synthetic dataset. The result also suggests that the AWV completion pattern may be reasonably captured by linear relationships involving features such as engagement score, prior AWV history, PCP attribution, SDOH risk, and utilization patterns.

These results should be interpreted as baseline model performance only. The models are not final. They establish a reference point for later work such as threshold tuning, model interpretation, and comparison with more advanced models.

Because this is synthetic data, the observed relationships reflect the assumptions built into the data-generation process. These results should not be interpreted as real-world causal evidence.